In [2]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os

In [6]:
inter_dir = '../02_INTERMEDIATE'
v1 = f'{inter_dir}/final_bamboo_parameters_v01.gpkg'
v10 = f'{inter_dir}/final_bamboo_parameters_v10.gpkg'

In [7]:
if os.path.exists(inter_dir):
    df_v1 = gpd.read_file(v1)
    df_v10 = gpd.read_file(v10)
else:
    print('doublecheck path')

In [8]:
df_v1.columns

Index(['psp', 'year', 'date_of_collection', 'clump_id', 'total_culms',
       'total_agb', 'max_culm_diameter_cm', 'avg_culm_diameter_cm',
       'min_culm_diameter_cm', 'median_culm_diameter_cm', 'max_culm_agb',
       'avg_culm_agb', 'min_culm_agb', 'median_culm_agb', 'hmin', 'hmax',
       'hmean', 'hstd', 'hmedian', 'total_points', 'canopy_volume_m3',
       'gap_fraction', 'canopy_area', 'val_height', 'val_canopy_diameter',
       'geometry'],
      dtype='str')

In [9]:
df_v10.columns

Index(['psp', 'year', 'date_of_collection', 'clump_id', 'total_culms',
       'total_agb', 'max_culm_diameter_cm', 'avg_culm_diameter_cm',
       'min_culm_diameter_cm', 'median_culm_diameter_cm', 'max_culm_agb',
       'avg_culm_agb', 'min_culm_agb', 'median_culm_agb', 'hmin', 'hmax',
       'hmean', 'hstd', 'hmedian', 'total_points', 'canopy_volume_m3',
       'gap_fraction', 'canopy_area', 'val_height', 'val_canopy_diameter',
       'breast_height_points', 'breast_height_valid_points',
       'breast_height_filtered_points', 'breast_height_cleaning_distance',
       'breast_height_area', 'breast_height_perimeter_m',
       'breast_height_diameter', 'bh_outside_area_m2', 'geometry'],
      dtype='str')

In [11]:
assert len(df_v10) == len(df_v1)

In [12]:
# ---------------------------------------------------------
# 1. Define the columns that uniquely identify a bamboo clump
# ---------------------------------------------------------

key_cols = ['psp', 'year', 'date_of_collection', 'clump_id']

value_cols = ['val_height', 'val_canopy_diameter']


# ---------------------------------------------------------
# 2. Make copies so the original dataframes are preserved
# ---------------------------------------------------------

df_v1_update = df_v1.copy()
df_v10_check = df_v10.copy()


# ---------------------------------------------------------
# 3. Make sure the key columns have compatible data types
# ---------------------------------------------------------

df_v1_update['psp'] = df_v1_update['psp'].astype(str).str.strip()
df_v10_check['psp'] = df_v10_check['psp'].astype(str).str.strip()

df_v1_update['clump_id'] = df_v1_update['clump_id'].astype(str).str.strip()
df_v10_check['clump_id'] = df_v10_check['clump_id'].astype(str).str.strip()

df_v1_update['year'] = pd.to_numeric(df_v1_update['year'], errors='coerce')
df_v10_check['year'] = pd.to_numeric(df_v10_check['year'], errors='coerce')

df_v1_update['date_of_collection'] = pd.to_datetime(
    df_v1_update['date_of_collection'],
    errors='coerce'
)

df_v10_check['date_of_collection'] = pd.to_datetime(
    df_v10_check['date_of_collection'],
    errors='coerce'
)


# ---------------------------------------------------------
# 4. Check for missing values in the matching columns
# ---------------------------------------------------------

print("Missing key values in df_v1:")
print(df_v1_update[key_cols].isna().sum())

print("\nMissing key values in df_v10:")
print(df_v10_check[key_cols].isna().sum())


# ---------------------------------------------------------
# 5. Check for duplicate clump keys
# ---------------------------------------------------------

duplicates_v1 = df_v1_update[
    df_v1_update.duplicated(key_cols, keep=False)
].sort_values(key_cols)

duplicates_v10 = df_v10_check[
    df_v10_check.duplicated(key_cols, keep=False)
].sort_values(key_cols)

print("\nDuplicate keys in df_v1:", len(duplicates_v1))
print("Duplicate keys in df_v10:", len(duplicates_v10))

if len(duplicates_v1) > 0:
    print("\nDuplicates in df_v1:")
    display(duplicates_v1[key_cols + value_cols])

if len(duplicates_v10) > 0:
    print("\nDuplicates in df_v10:")
    display(duplicates_v10[key_cols + value_cols])

Missing key values in df_v1:
psp                   0
year                  0
date_of_collection    0
clump_id              0
dtype: int64

Missing key values in df_v10:
psp                   0
year                  0
date_of_collection    0
clump_id              0
dtype: int64

Duplicate keys in df_v1: 0
Duplicate keys in df_v10: 0


In [13]:
# ---------------------------------------------------------
# 6. Check which df_v1 records have a matching df_v10 record
# ---------------------------------------------------------

match_check = df_v1_update[key_cols].merge(
    df_v10_check[key_cols],
    on=key_cols,
    how='left',
    indicator=True
)

print(match_check['_merge'].value_counts())

missing_in_v10 = match_check[
    match_check['_merge'] == 'left_only'
]

print("\nRecords in df_v1 with NO matching record in df_v10:")
display(missing_in_v10)

print("\nNumber of unmatched df_v1 records:", len(missing_in_v10))

_merge
both          225
left_only       0
right_only      0
Name: count, dtype: int64

Records in df_v1 with NO matching record in df_v10:


,psp,year,date_of_collection,clump_id,_merge



Number of unmatched df_v1 records: 0


In [14]:
# ---------------------------------------------------------
# 7. Check for df_v10 records that don't exist in df_v1
# ---------------------------------------------------------

reverse_check = df_v10_check[key_cols].merge(
    df_v1_update[key_cols],
    on=key_cols,
    how='left',
    indicator=True
)

missing_in_v1 = reverse_check[
    reverse_check['_merge'] == 'left_only'
]

print("Records in df_v10 with NO matching record in df_v1:")
display(missing_in_v1)

print("\nNumber of df_v10 records without a df_v1 match:", len(missing_in_v1))

Records in df_v10 with NO matching record in df_v1:


,psp,year,date_of_collection,clump_id,_merge



Number of df_v10 records without a df_v1 match: 0


In [15]:
# ---------------------------------------------------------
# 8. Create the lookup table from df_v10
# ---------------------------------------------------------

lookup = df_v10_check[
    key_cols + value_cols
].copy()

lookup.head()

,psp,year,date_of_collection,clump_id,val_height,val_canopy_diameter
0,1a-APM-01,2025,2025-04-24,CLUMP #1,NaN,NaN
1,1a-APM-01,2026,2026-04-02,CLUMP #1,NaN,NaN
2,1a-APM-01,2025,2025-04-24,CLUMP #2,NaN,NaN
3,1a-APM-01,2026,2026-04-02,CLUMP #2,NaN,NaN
4,1a-APM-01,2025,2025-04-24,CLUMP #3,NaN,NaN


In [16]:
# ---------------------------------------------------------
# 9. Merge df_v10 values into df_v1
# ---------------------------------------------------------

df_v1_merged = df_v1_update.merge(
    lookup,
    on=key_cols,
    how='left',
    suffixes=('', '_v10')
)

print(df_v1_merged.shape)

(225, 28)


In [17]:
# ---------------------------------------------------------
# 10. Compare the old and new values
# ---------------------------------------------------------

height_changed = (
    df_v1_merged['val_height'].notna() &
    df_v1_merged['val_height_v10'].notna() &
    ~np.isclose(
        df_v1_merged['val_height'],
        df_v1_merged['val_height_v10'],
        equal_nan=True
    )
)

diameter_changed = (
    df_v1_merged['val_canopy_diameter'].notna() &
    df_v1_merged['val_canopy_diameter_v10'].notna() &
    ~np.isclose(
        df_v1_merged['val_canopy_diameter'],
        df_v1_merged['val_canopy_diameter_v10'],
        equal_nan=True
    )
)

print("val_height values that would change:", height_changed.sum())
print("val_canopy_diameter values that would change:", diameter_changed.sum())

val_height values that would change: 8
val_canopy_diameter values that would change: 0


In [18]:
changes = df_v1_merged[
    height_changed | diameter_changed
][
    key_cols +
    [
        'val_height',
        'val_height_v10',
        'val_canopy_diameter',
        'val_canopy_diameter_v10'
    ]
].copy()

display(changes)

,psp,year,date_of_collection,clump_id,val_height,val_height_v10,val_canopy_diameter,val_canopy_diameter_v10
58,1a-APM-02,2025,2025-04-25,CLUMP #15,9.1,8.4,NaN,NaN
59,1a-APM-02,2026,2026-05-03,CLUMP #15,9.1,8.4,NaN,NaN
183,1a-GB-05,2025,2025-06-18,CLUMP #2,5.4,5.9,7.4,7.4
184,1a-GB-05,2026,2026-03-13,CLUMP #2,5.4,5.9,7.4,7.4
185,1a-GB-05,2026,2026-06-24,CLUMP #2,5.4,5.9,7.4,7.4
189,1a-GB-05,2025,2025-06-18,CLUMP #4,8.0,7.0,6.4,6.4
190,1a-GB-05,2026,2026-03-13,CLUMP #4,8.0,7.0,6.4,6.4
191,1a-GB-05,2026,2026-06-24,CLUMP #4,8.0,7.0,6.4,6.4


In [ ]:
# ---------------------------------------------------------
# 11. Update ONLY the two requested columns
# ---------------------------------------------------------

df_v1_merged['val_height'] = df_v1_merged['val_height_v10']

df_v1_merged['val_canopy_diameter'] = df_v1_merged['val_canopy_diameter_v10']


# ---------------------------------------------------------
# 12. Remove the temporary columns
# ---------------------------------------------------------

df_v1_updated = df_v1_merged.drop(
    columns=['val_height_v10', 'val_canopy_diameter_v10']
)

In [20]:
df_v1_updated.columns

Index(['psp', 'year', 'date_of_collection', 'clump_id', 'total_culms',
       'total_agb', 'max_culm_diameter_cm', 'avg_culm_diameter_cm',
       'min_culm_diameter_cm', 'median_culm_diameter_cm', 'max_culm_agb',
       'avg_culm_agb', 'min_culm_agb', 'median_culm_agb', 'hmin', 'hmax',
       'hmean', 'hstd', 'hmedian', 'total_points', 'canopy_volume_m3',
       'gap_fraction', 'canopy_area', 'val_height', 'val_canopy_diameter',
       'geometry'],
      dtype='str')

In [22]:
assert len(df_v1.columns) == len(df_v1_updated.columns)

In [23]:
df_v1_updated.to_file(f'{inter_dir}/final_bamboo_parameters_DV001.gpkg', driver='GPKG')